This file is used to create judgements of all responses generated by the models. The model used in for this will be Gemini 2.5

In [18]:
import google.generativeai as genai
from groq import Groq
import json
from openai import OpenAI
import cohere
from mistralai import Mistral
import json
import os

### Inference function calling API

In [71]:
# Data generation function using API

def create_synthetic_data(model, input_path, output_path, split=None):

    if model.startswith("gemini"):
        # Gemini API
        with open("/Users/maxschaffelder/Desktop/Thesis/Keys/gemini_key.txt", "r") as f:
            key = f.read().strip()

        # Configure Gemini API
        genai.configure(api_key=key)

    elif model == "llama-3.1-8b-instant" or model == "gemma2-9b-it":
        # Groq API
        with open("/Users/maxschaffelder/Desktop/Thesis/Keys/groq_key.txt", "r") as f:
            key = f.read().strip()

        client = Groq(api_key=key)

    elif model == "meta-llama/Meta-Llama-3.1-70B-Instruct" or model == "Qwen/Qwen2.5-72B-Instruct" or model == "deepseek-ai/DeepSeek-V3" or model == "meta-llama/Meta-Llama-3.1-405B-Instruct":
        # Deepinfra API
        with open("/Users/maxschaffelder/Desktop/Thesis/Keys/deepinfra_key.txt", "r") as f:
            key = f.read().strip()

        client = OpenAI(
            api_key=key,
            base_url="https://api.deepinfra.com/v1/openai",
        )

    elif model == "command-r-plus":
        # Cohere API
        with open("/Users/maxschaffelder/Desktop/Thesis/Keys/cohere_key_paid.txt", "r") as f:
            key = f.read().strip()
        
        client = cohere.ClientV2(key)

    elif model == "mistral-large-latest":
        # Mistral API
        with open("/Users/maxschaffelder/Desktop/Thesis/Keys/mistral_key.txt", "r") as f:
            key = f.read().strip()

        client = Mistral(api_key=key)

    elif model == "gpt-4o":
        # OpenAI API
        with open("/Users/maxschaffelder/Desktop/Thesis/Keys/openai_key.txt", "r") as f:
            key = f.read().strip()

        client = OpenAI(
            api_key=key,
        )



    # Read lines from the input file
    with open(input_path, "r", encoding="utf-8") as fin:
        all_lines = fin.readlines()

    # If `split` is set, slice accordingly
    if split is not None:
        all_lines = all_lines[split[0]:split[1]] 

    # Open output in append mode
    with open(output_path, "a", encoding="utf-8") as fout:
        for i, line in enumerate(all_lines):
            try:
                data = json.loads(line)
                instruction = data["judge_input"]


                if model.startswith("gemini"):

                    # Call Gemini API
                    generation_config = {"temperature": 0.7, "top_p": 0.9}
                    #model_instance = genai.GenerativeModel(model, system_instruction="You are a helpful assistant.")
                    #response = model_instance.generate_content(instruction, generation_config=generation_config)

                    safety_settings = [ 
                        {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"}, 
                        {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"}, 
                        {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"}, 
                        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
                        ]
                    
                    model_instance = genai.GenerativeModel(model, system_instruction="You are a helpful assistant.")
                    response = model_instance.generate_content(instruction, generation_config=generation_config, safety_settings=safety_settings)



                    if response.candidates:
                        candidate = response.candidates[0]
                        if candidate.content and candidate.content.parts:
                            generated_text = candidate.content.parts[0].text
                        else:
                            generated_text = f"Error: No content parts. Candidate: {candidate}"
                            print(f"Debug - Full response: {response}")
                            break
                    else:
                        generated_text = f"Error: No candidates. Response: {response}"
                        print(f"Debug - Full response: {response}")
                        if hasattr(response, 'prompt_feedback'):
                            print(f"Debug - Prompt Feedback: {response.prompt_feedback}")
                        break

                    #if response.candidates:
                    #    generated_text = response.candidates[0].content.parts[0].text

                    #else:
                    #    generated_text = "Error: No response received"


                elif model == "command-r-plus":

                    response = client.chat(
                        model=model, 
                        messages=[
                            {"role": "system", "content": "You are a helpful assistant."},
                            {"role": "user", "content": instruction}],
                        max_tokens=1024,
                        temperature=0.7,
                        p=0.9
                    )

                    generated_text = response.message.content[0].text

                elif model == "mistral-large-latest":
                    response = client.chat.complete(
                        model = model,
                        messages = [
                            {"role": "system", "content": "You are a helpful assistant."},
                            {"role": "user", "content": instruction}],
                            max_tokens=1024,
                            temperature=0.7,
                            top_p=0.9
                    )
                    generated_text = response.choices[0].message.content

                else: 
                    chat_completion = client.chat.completions.create(
                        messages=[
                            {"role": "system", "content": "You are a helpful assistant."},
                            {"role": "user", "content": instruction}],
                        model=model,
                        temperature=0.7,
                        top_p=0.9,
                        max_tokens=1024
                    )
                    generated_text = chat_completion.choices[0].message.content
                    
                data[f"judgement"] = generated_text
                data[f"judge_model"] = model


                # Write out the updated data immediately
                fout.write(json.dumps(data, ensure_ascii=False))
                fout.write("\n")

                # Every 100 lines, explicitly flush to disk
                if i % 100 == 0:
                    fout.flush()

            except Exception as e:
                print(f"Error on line {i}: {e}")
                break  # Adjust behavior as needed

    print("Done generating synthetic data.")


### Loading judge input data

In [69]:
refusalbench_judge_input = {}

# Store file paths from refusalbench_only folder
refusalbench_only_path = '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_only'
for filename in os.listdir(refusalbench_only_path):
    if filename.endswith('.jsonl'):
        key = filename.replace('.jsonl', '').replace("judge_input_", "")
        filepath = os.path.join(refusalbench_only_path, filename)
        refusalbench_judge_input[key] = filepath
        print(f"Stored path for {key}")

refusalbench_jailbreak_judge_input = {}

# Store file paths from refusalbench_jailbreak folder
refusalbench_jailbreak_path = '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_jailbreak'
for filename in os.listdir(refusalbench_jailbreak_path):
    if filename.endswith('.jsonl'):
        key = filename.replace('.jsonl', '').replace("judge_input_", "")
        filepath = os.path.join(refusalbench_jailbreak_path, filename)
        refusalbench_jailbreak_judge_input[key] = filepath
        print(f"Stored path for {key}")


Stored path for refusalbench_8b_vanilla
Stored path for refusalbench_8b_single_source
Stored path for refusalbench_8b_human_source
Stored path for refusalbench_8b_multi_source
Stored path for refusalbench_jailbreak_8b_human_source
Stored path for refusalbench_jailbreak_8b_vanilla
Stored path for refusalbench_jailbreak_8b_multi_source
Stored path for refusalbench_jailbreak_8b_single_source


In [33]:
all_inputs = ""
lenght_in_words_inputs = 0
for name, input_path in refusalbench_judge_input.items():
    with open(input_path, "r") as f:
        data = [json.loads(line) for line in f]

    for line in data:
        lenght_in_words_inputs += len(line["judge_input"].split())

for name, input_path in refusalbench_jailbreak_judge_input.items():
    with open(input_path, "r") as f:
        data = [json.loads(line) for line in f]

    for line in data:
        lenght_in_words_inputs += len(line["judge_input"].split())

print(lenght_in_words_inputs) # ~13,305,415.76 input tokens

10004072


### Generate judgements using Gemini-2.5-flash

In [41]:
refusalbench_jailbreak_judge_input = {
    'refusalbench_jailbreak_8b_human_source': '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_jailbreak/judge_input_refusalbench_jailbreak_8b_human_source.jsonl',
    'refusalbench_jailbreak_8b_vanilla': '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_jailbreak/judge_input_refusalbench_jailbreak_8b_vanilla.jsonl',
    'refusalbench_jailbreak_8b_multi_source': '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_jailbreak/judge_input_refusalbench_jailbreak_8b_multi_source.jsonl',
    'refusalbench_jailbreak_8b_single_source': '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_jailbreak/judge_input_refusalbench_jailbreak_8b_single_source.jsonl'
    }

In [73]:
from_line = 469

# 133 didn't work for refusalbench_jailbreak_8b_human_source, skipped it
# 448 didn't work for refusalbench_jailbreak_8b_human_source, skipped it
# 464 didn't work
# 465 didn't work
# 467 didn't work
# 469 didn't work

model = "gemini-2.5-flash-preview-05-20"

name  = "refusalbench_jailbreak_8b_human_source"
input_path = '/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/inputs/refusalbench_jailbreak/judge_input_refusalbench_jailbreak_8b_human_source.jsonl'
output_path = f"/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/refusalbench_jailbreak/{name}_judged.jsonl"
create_synthetic_data(model, input_path, output_path, split=[from_line, None])

#for name, input_path in refusalbench_jailbreak_judge_input.items():  
#    output_path = f"/Users/maxschaffelder/Desktop/Thesis/data/exp_2/judge/outputs/refusalbench_jailbreak/{name}_judged.jsonl"
#    create_synthetic_data(model, input_path, output_path, split=[from_line, None])
    

Debug - Full response: response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "prompt_feedback": {
        "block_reason": "OTHER"
      },
      "usage_metadata": {
        "prompt_token_count": 1777,
        "total_token_count": 1777
      }
    }),
),
error=BlockedPromptException(prompt_feedback {
  block_reason: OTHER
}
usage_metadata {
  prompt_token_count: 1777
  total_token_count: 1777
}
)
Debug - Prompt Feedback: block_reason: OTHER

Done generating synthetic data.


In [12]:
# Check available Gemini models
with open("/Users/maxschaffelder/Desktop/Thesis/Keys/gemini_key.txt", "r") as f:
    key = f.read().strip()

genai.configure(api_key=key)

# List available models
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"Model: {model.name}")
        print(f"  Display name: {model.display_name}")
        print(f"  Description: {model.description}")
        print("---")

Model: models/gemini-1.0-pro-vision-latest
  Display name: Gemini 1.0 Pro Vision
  Description: The original Gemini 1.0 Pro Vision model version which was optimized for image understanding. Gemini 1.0 Pro Vision was deprecated on July 12, 2024. Move to a newer Gemini version.
---
Model: models/gemini-pro-vision
  Display name: Gemini 1.0 Pro Vision
  Description: The original Gemini 1.0 Pro Vision model version which was optimized for image understanding. Gemini 1.0 Pro Vision was deprecated on July 12, 2024. Move to a newer Gemini version.
---
Model: models/gemini-1.5-pro-latest
  Display name: Gemini 1.5 Pro Latest
  Description: Alias that points to the most recent production (non-experimental) release of Gemini 1.5 Pro, our mid-size multimodal model that supports up to 2 million tokens.
---
Model: models/gemini-1.5-pro-001
  Display name: Gemini 1.5 Pro 001
  Description: Stable version of Gemini 1.5 Pro, our mid-size multimodal model that supports up to 2 million tokens, released 